# Notebook 03 — Reaction Path Analysis

Computes and visualises free energy diagrams for three NRR pathways and HER
across all nine bimetallic catalyst systems.

**Pathways covered:**
- Associative alternating
- Associative distal
- Dissociative
- Hydrogen evolution reaction (HER, competing pathway)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src import (
    SYSTEMS_LABEL_MAP,
    SYSTEM_COLORS,
    compute_reaction_path_assoc_alternating,
    compute_reaction_path_assoc_distal,
    compute_reaction_path_dissociative,
    compute_her_path,
    compute_step_barriers,
    find_top_barriers,
)

## Load DFT energy data

In [ ]:
df = pd.read_csv('../data/corrected_df.csv')
df.head()

## System and colour settings

In [ ]:
systems    = ['0_CuNi','1_Ag_subCu','2_Au_subCu','6_Pd_subNi',
               '7_Pt_subNi','9_Fe_subNi','11_Pd_subCu','13_Ag_subNi','14_Au_subNi']
legends    = [SYSTEMS_LABEL_MAP[s] for s in systems]
colors     = [SYSTEM_COLORS[SYSTEMS_LABEL_MAP[s]] for s in systems]
site_number = 1

## Compute reaction paths

In [ ]:
path_assoc_alt  = compute_reaction_path_assoc_alternating(df, site_number=site_number)
path_assoc_dist = compute_reaction_path_assoc_distal(df, site_number=site_number)
path_dissoc     = compute_reaction_path_dissociative(df, site_number=site_number)
path_her        = compute_her_path(df, site_number=site_number)

print('Associative alternating path — energy levels:')
path_assoc_alt

## Top rate-limiting steps

In [ ]:
barriers_alt = compute_step_barriers(path_assoc_alt, system_cols=systems)
top_alt      = find_top_barriers(barriers_alt, n=3)

print('Top 3 barriers — associative alternating pathway:')
top_alt.sort_values('barrier_1_val')

In [ ]:
barriers_her = compute_step_barriers(path_her, system_cols=systems)
top_her      = find_top_barriers(barriers_her, n=1)

print('HER limiting step barrier per catalyst:')
top_her.sort_values('barrier_1_val')

## Visualise — Associative alternating pathway

> **Note:** The `plot_reaction_energy_diagram` function requires a `test_path` DataFrame
> (interpolated connector values between plateaus). Build it from `path_assoc_alt`
> by repeating each row 5 times (see original `correlations_reaction_path.ipynb`).

In [ ]:
# Build interpolated test_path for plateau connectors
test_path = path_assoc_alt[systems].copy()
expanded = []
for idx in test_path.index:
    for _ in range(5):
        expanded.append(test_path.loc[idx])
test_path_expanded = pd.DataFrame(expanded).reset_index(drop=True)

xticks = []
for step in path_assoc_alt['steps']:
    xticks += ['', '', step, '', '']

from src.visualization import plot_reaction_energy_diagram

fig = plot_reaction_energy_diagram(
    path_df=path_assoc_alt,
    test_path=test_path_expanded,
    xticks=xticks,
    systems=systems,
    legends=legends,
    colors=colors,
    title=f'Associative Alternating — Site {site_number}',
    figsize=(35, 20),
    # save_path='../figures/assoc_alt_path.pdf',
)
plt.show()